# Week 6: Spark Architecture and Data Processing

## Objective

The objective of this assignment is to understand Apache Spark architecture and perform efficient data processing using PySpark. This includes reading datasets, applying transformations and actions, handling schemas, filtering data, working with CSV and Parquet files, and building a simple data processing pipeline.


## Learning Outcomes

By completing this assignment, we will be able to:

- Understand Spark Architecture (Driver, Cluster Manager, Executors)
- Learn Lazy Evaluation and DAG (Lineage Graph)
- Read CSV and Parquet files with proper schema handling
- Perform filtering and column selection
- Rename columns and change data types
- Add new calculated columns
- Understand Transformations and Actions
- Learn Shuffle and Predicate Pushdown concepts
- Handle null values efficiently
- Build a complete Spark pipeline (Read → Transform → Filter → Write)
- Save processed data into CSV and Parquet formats
- Follow Spark best practices for large datasets


# Step 1: Install PySpark

In this step, we install the PySpark library, which allows Python to interact with Apache Spark for distributed data processing.

In [2]:
# Install PySpark

!pip install pyspark

# Step 2: Import Required Libraries

The following libraries are required for creating a Spark session and performing DataFrame operations.

In [3]:
# Import SparkSession for creating a Spark application
from pyspark.sql import SparkSession

# Import commonly used SQL functions
from pyspark.sql.functions import *

# Import Spark SQL data types
from pyspark.sql.types import *

# Step 3: Create Spark Session

A Spark Session is the entry point to every Spark application. It allows us to create DataFrames, read data, perform transformations, and execute Spark jobs.

In [4]:
# Create a Spark Session
spark = SparkSession.builder \
    .appName("Week6_Spark_Assignment") \
    .getOrCreate()

# Display confirmation message
print("Spark Session Created Successfully!")

Spark Session Created Successfully!


# Step 4: Confirm Spark Installation

Check the Spark version installed to ensure that the environment is set up correctly.

In [5]:
# Display Spark version

print("Spark Version:", spark.version)

Spark Version: 4.0.3


# Week 6 Assignment Questions

# Q1. Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

## Answer
Spark is an open-source big data processing engine that uses the **Master-Slave architecture** in order to process datasets efficiently.

### 1. Driver
The **Driver** is the primary program running in a Spark Application. It initiates the Spark Session, transforms the user code into tasks, forms the DAG (Directed Acyclic Graph), schedules tasks, and finally gathers the results generated from the Executors.

### 2. Cluster Manager
The **Cluster Manager** performs the function of allocating resources available in the cluster. It manages the allocation of CPU and memory resources and spawns Executors in the worker nodes. Examples of a Cluster Manager include Standalone Cluster Manager, YARN, Mesos, and Kubernetes.

### 3. Executor
**Executors** are worker threads that actually do the job of executing the tasks that are assigned by the Driver. They do data partition processing, computation of the data, store the intermediary results in memory, and send back the output to the driver.


### Insight
The Driver is the brain of the Spark program, the Cluster Manager provides computing resources, while the Executors do the data processing work.

# Q2. How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets?

## Answer
Apache Spark uses the **Lazy Evaluation** technique, which implies that **transformations are not executed right away**. Apache Spark records all transformations and creates a **Directed Acyclic Graph (DAG)**, which is also referred to as the **Lineage Graph**.

Execution takes place only after calling the **action** (`show()`, `collect()`, `count()`, etc.). Prior to execution, Apache Spark runs Catalyst Optimizer, which analyses the DAG and forms the execution plan for the job.

This leads to better performance, as unnecessary computation is minimized, disk I/O and network communication are reduced, and several transformations are combined into one execution.


## How Lazy Evaluation Takes Place

1. Loading the dataset.
2. Applying one or several transformations (`select()`, `filter()`, `withColumn()`).
3. Recording transformations but not executing them.
4. Running an action (`show()`, `count()`, `write()`, etc.).
5. Execution plan for the DAG creation and running the job.


## Benefits of Lazy Evaluation Technique

- Better execution performance.
- Minimizes unnecessary computations.
- Decreases disk I/O and network communication.
- Optimized execution of queries due to DAG.
- Efficient utilization of resources on the cluster.

In [11]:
# Create a sample DataFrame
data = [
    (1, "Laptop", 65000),
    (2, "Mobile", 25000),
    (3, "Tablet", 18000),
    (4, "Monitor", 12000)
]

columns = ["product_id", "product_name", "price"]

df = spark.createDataFrame(data, columns)


# Lazy Transformations

filtered_df = df.filter(df.price > 20000)
selected_df = filtered_df.select("product_name", "price")

# At this point, Spark has only created a logical execution plan.
# No computation has been performed yet.

print("Transformations created successfully.")
print("No execution has happened yet!")


# Action
# This triggers the execution.


selected_df.show()

Transformations created successfully.
No execution has happened yet!
+------------+-----+
|product_name|price|
+------------+-----+
|      Laptop|65000|
|      Mobile|25000|
+------------+-----+



### Insight

Spark's Lazy Evaluation delays execution until an Action is performed. This allows Spark to optimize the complete sequence of transformations using a Directed Acyclic Graph (DAG), reducing unnecessary computations and improving performance, especially for large-scale datasets.

# Q3. Write a Spark command to read a CSV file ensuring the first row is treated as a header and inferSchema is enabled.

## Answer
In PySpark, the `spark.read.csv()` function is used to load CSV files into a DataFrame. The `header=True` option treats the first row as column names, while `inferSchema=True` automatically detects the data type of each column.

This reduces manual effort and improves data processing efficiency.

In [12]:
# Read the products.csv file into a Spark DataFrame

df = spark.read.csv(
    "products.csv",      # Path to the CSV file
    header=True,         # Treat the first row as column headers
    inferSchema=True     # Automatically infer the data types
)

# Display the first 5 rows
df.show(5)

# Display the schema of the DataFrame
df.printSchema()

+----------+---------------+--------------+-------+------+------+
|product_id|   product_name|      category|  brand| price|rating|
+----------+---------------+--------------+-------+------+------+
|   P000001|       Astra Be|      Clothing|  Astra|157.89|  4.08|
|   P000002|NeoTech Someone|     Groceries|NeoTech| 21.46|  3.87|
|   P000003|   Acme Discuss|        Sports|   Acme|265.37|  3.46|
|   P000004|   Nimbus South|   Electronics| Nimbus|541.41|  4.14|
|   P000005|  Astra Capital|Home & Kitchen|  Astra| 198.0|  3.97|
+----------+---------------+--------------+-------+------+------+
only showing top 5 rows
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- rating: double (nullable = true)



# Q4. What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar), and why does it matter for performance?

## Answer

CSV (Comma-Separated Values) is a **row-based** file format, whereas Parquet is a **columnar** file format.

In a CSV file, data is stored row by row. This makes it simple and human-readable but less efficient for analytical queries because Spark has to read every column, even if only a few columns are needed.

Parquet stores data column by column. When only specific columns are required, Spark reads only those columns instead of the entire dataset. This significantly reduces disk I/O, memory usage, and processing time.

Additionally, Parquet supports compression and stores schema information, making it more suitable for big data processing.


## Difference between CSV and Parquet

| Feature | CSV | Parquet |
|----------|-----|----------|
| Storage Format | Row-based | Columnar |
| File Size | Larger | Smaller (Compressed) |
| Schema Support | No | Yes |
| Read Performance | Slower | Faster |
| Compression | No | Yes |
| Best Use Case | Data exchange and simple storage | Big data analytics and Spark processing |

## Insight

CSV files are easy to read and widely supported, making them suitable for data sharing. However, Parquet provides much better performance in Spark because it stores data in a columnar format, supports compression, and enables faster query execution on large datasets.


# Q5. Given a DataFrame `df`, write a query to select the columns `product_id` and `price` where the `category` is **'Electronics'**.

## Answer

The `select()` function is used to retrieve specific columns from a DataFrame, while the `filter()` function is used to extract rows that satisfy a given condition.

In this query, we select only the `product_id` and `price` columns for products whose category is **Electronics**.

In [8]:
# Select only the required columns after filtering Electronics category

electronics_df = df.filter(df.category == "Electronics") \
                   .select("product_id", "price")

# Display the filtered DataFrame
electronics_df.show()

+----------+-------+
|product_id|  price|
+----------+-------+
|   P000004| 541.41|
|   P000012|1299.71|
|   P000014|  99.56|
|   P000021|1772.36|
|   P000035| 438.46|
|   P000051| 840.01|
|   P000055| 949.73|
|   P000058| 788.21|
|   P000070| 802.45|
|   P000114| 805.15|
|   P000136|2169.98|
|   P000160| 259.69|
|   P000161| 701.36|
|   P000165| 295.27|
|   P000174| 960.74|
|   P000186| 231.15|
|   P000192| 455.43|
|   P000198|1352.63|
|   P000212| 246.31|
|   P000218| 234.99|
+----------+-------+
only showing top 20 rows


## Insight

Filtering data before selecting columns reduces the amount of data processed, which improves efficiency. Spark performs this operation lazily and executes it only when an action such as `show()` is called.

# Q6. Write the code to revise a DataFrame by renaming the column `old_name` to `new_name` and casting the `price` column from a String to a Double.

## Answer

In PySpark, the `withColumnRenamed()` function is used to rename an existing column, while the `cast()` function is used to change the data type of a column.

In this example, the `product_name` column is renamed to `product_title`, and the `price` column is cast to the `Double` data type. Although the `price` column is already of type `Double` in our dataset, this demonstrates the correct syntax for type conversion.

In [9]:
# Import the col() function for column operations
from pyspark.sql.functions import col

# Rename the 'product_name' column to 'product_title'
# Cast the 'price' column to Double type

df = df.withColumnRenamed("product_name", "product_title") \
       .withColumn("price", col("price").cast("double"))

# Display the first 5 rows of the updated DataFrame
df.show(5)

# Display the updated schema
df.printSchema()

+----------+---------------+--------------+-------+------+------+
|product_id|  product_title|      category|  brand| price|rating|
+----------+---------------+--------------+-------+------+------+
|   P000001|       Astra Be|      Clothing|  Astra|157.89|  4.08|
|   P000002|NeoTech Someone|     Groceries|NeoTech| 21.46|  3.87|
|   P000003|   Acme Discuss|        Sports|   Acme|265.37|  3.46|
|   P000004|   Nimbus South|   Electronics| Nimbus|541.41|  4.14|
|   P000005|  Astra Capital|Home & Kitchen|  Astra| 198.0|  3.97|
+----------+---------------+--------------+-------+------+------+
only showing top 5 rows
root
 |-- product_id: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- rating: double (nullable = true)



## Insight

Renaming columns improves readability, while casting ensures that columns have the correct data type for analysis and calculations. Proper schema management is an essential step in data preprocessing.

# Q7. How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

## Answer
Apache Spark maintains a **Lineage Graph**, also known as a **Directed Acyclic Graph (DAG)**, which records the sequence of transformations applied to a dataset. Instead of storing multiple copies of intermediate data, Spark remembers how the data was created.

If a worker node (Executor) fails and some data partitions are lost, Spark uses the Lineage Graph to identify the missing partitions and automatically recomputes only those partitions from the original data. It does not re-execute the entire application.

This approach provides efficient fault tolerance while reducing storage overhead and improving reliability in distributed data processing.

## How Fault Tolerance Works
1. Read the input data.
2. Apply one or more transformations.
3. Spark records these transformations in the Lineage Graph (DAG).
4. If an Executor fails, the lost partition is identified.
5. Spark recomputes only the missing partition using the recorded transformations.
6. The remaining partitions continue processing without interruption.

## Insight
Spark achieves fault tolerance through its Lineage Graph (DAG), which stores the sequence of transformations instead of duplicating data. If a worker node fails, Spark efficiently rebuilds only the lost partitions, making distributed processing reliable and scalable.

# Q8. Write a query to filter a DataFrame `df_orders` for rows where the status is **'Completed'** AND the amount is greater than **1000**.

## Answer

The `filter()` function is used to retrieve rows that satisfy one or more conditions. In this query, the DataFrame is filtered to display only those orders where the **order_status** is **Completed** and the **total_amount** is greater than **1000**.

**Note:** In our dataset, the columns are named **order_status** and **total_amount** instead of **status** and **amount**.

In [9]:
# Filter orders where the order status is 'Completed'
# and the total amount is greater than 1000

filtered_orders = df_orders.filter(
    (df_orders.order_status == "completed") &
    (df_orders.total_amount > 1000)
)

# Display the filtered records
filtered_orders.show(5)

+---------+-------+--------------------+------------+------------+
| order_id|user_id|          order_date|order_status|total_amount|
+---------+-------+--------------------+------------+------------+
|O00000002|U003247|2025-04-15 01:18:...|   completed|     1666.85|
|O00000006|U003449|2024-10-13 09:02:...|   completed|     2258.34|
|O00000063|U001271|2024-02-24 19:27:...|   completed|     1087.18|
|O00000082|U002630|2024-07-17 14:13:...|   completed|     1206.21|
|O00000091|U006570|2024-01-23 19:27:...|   completed|     1707.41|
+---------+-------+--------------------+------------+------------+
only showing top 5 rows


# Q9. Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

## Answer

**Predicate Pushdown** is an optimization technique used by Spark when reading Parquet files. Instead of loading the entire dataset into memory, Spark pushes the filtering condition down to the Parquet storage layer.

As a result, only the rows that satisfy the filter condition are read from the disk, while the remaining data is skipped. This significantly reduces disk I/O, memory usage, and query execution time.

Predicate Pushdown is supported by Parquet because it stores metadata such as column statistics, allowing Spark to determine which data blocks need to be read.



## Advantages of Predicate Pushdown

- Reduces the amount of data read from disk.
- Improves query performance.
- Decreases memory usage.
- Minimizes disk I/O operations.
- Optimizes processing of large datasets.

## Insight

Predicate Pushdown allows Spark to read only the required data from Parquet files instead of scanning the entire dataset. This optimization improves performance, especially when working with large-scale data.

# Q10. Write a code snippet to add a new column `final_price` which is the `price` multiplied by **1.18** (18% tax).

## Answer

The `withColumn()` function is used to create a new column or modify an existing column in a DataFrame. In this example, a new column named **final_price** is added by multiplying the **price** column by **1.18**, representing an 18% tax.

In [13]:
# Add a new column 'final_price' by applying 18% tax to the price

from pyspark.sql.functions import col

df = df.withColumn("final_price", col("price") * 1.18)

# Display the first 5 rows
df.show(5)

+----------+---------------+--------------+-------+------+------+------------------+
|product_id|   product_name|      category|  brand| price|rating|       final_price|
+----------+---------------+--------------+-------+------+------+------------------+
|   P000001|       Astra Be|      Clothing|  Astra|157.89|  4.08|186.31019999999998|
|   P000002|NeoTech Someone|     Groceries|NeoTech| 21.46|  3.87|           25.3228|
|   P000003|   Acme Discuss|        Sports|   Acme|265.37|  3.46|          313.1366|
|   P000004|   Nimbus South|   Electronics| Nimbus|541.41|  4.14|          638.8638|
|   P000005|  Astra Capital|Home & Kitchen|  Astra| 198.0|  3.97|            233.64|
+----------+---------------+--------------+-------+------+------+------------------+
only showing top 5 rows


## Insight

The `withColumn()` function is useful for creating derived columns without modifying the original dataset. It is commonly used in Spark for performing calculations, feature engineering, and data transformation.

# Q11. What is the difference between Transformations and Actions? Provide two examples of each.

## Answer

In Apache Spark, operations are divided into **Transformations** and **Actions**.

### Transformations

Transformations are operations that create a new DataFrame or RDD from an existing one. They are **lazy**, meaning Spark does not execute them immediately. Instead, Spark records these operations in the Directed Acyclic Graph (DAG) and executes them only when an Action is called.

**Examples:**
- `filter()`
- `select()`

### Actions

Actions are operations that trigger the execution of all previously defined transformations. They produce a result or return data to the Driver.

**Examples:**
- `show()`
- `count()`


## Difference between Transformations and Actions

| Transformations | Actions |
|-----------------|---------|
| Create a new DataFrame or RDD | Execute the transformations |
| Lazy execution | Immediate execution |
| Do not return the final result | Return or display the result |
| Build the DAG | Trigger the DAG execution |


# Q12. Write the Spark command to load a Parquet file from `"path/to/input"`, filter out any rows where `user_id` is null, and save the result as a CSV at `"path/to/output"`.

## Answer

In this question, we first load the Parquet file into a Spark DataFrame. Next, we remove rows where the `user_id` column contains null values using the `filter()` function. Finally, the filtered DataFrame is saved as a CSV file.

In [14]:
# Save the orders DataFrame as a Parquet file

df_orders.write.mode("overwrite").parquet("orders_parquet")

print("Parquet file created successfully.")

Parquet file created successfully.


In [15]:
# Read the Parquet file

parquet_df = spark.read.parquet("orders_parquet")

# Filter rows where user_id is not null

filtered_df = parquet_df.filter(parquet_df.user_id.isNotNull())

# Save the filtered DataFrame as a CSV file

filtered_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("filtered_orders_csv")

print("Filtered data saved successfully as CSV.")

Filtered data saved successfully as CSV.


In [16]:
# Display the filtered DataFrame

filtered_df.show(5)

+---------+-------+--------------------+------------+------------+
| order_id|user_id|          order_date|order_status|total_amount|
+---------+-------+--------------------+------------+------------+
|O00000001|U009310|2025-09-09 14:52:...|  processing|      689.66|
|O00000002|U003247|2025-04-15 01:18:...|   completed|     1666.85|
|O00000003|U007252|2025-04-27 15:37:...|  processing|      665.06|
|O00000004|U008986|2025-10-04 20:35:...|   cancelled|       689.5|
|O00000005|U008537|2024-11-13 08:15:...|   cancelled|       860.5|
+---------+-------+--------------------+------------+------------+
only showing top 5 rows


## Insight

Parquet is an efficient columnar storage format that improves query performance and reduces storage requirements. Filtering null values before saving the dataset ensures better data quality and prepares the data for further analysis.

# Q13. In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

## Answer

Apache Spark supports two deployment modes: **Client Mode** and **Cluster Mode**.

### Client Mode

In Client Mode, the **Driver** program runs on the local machine from where the Spark application is submitted. The Executors run on the cluster nodes and communicate with the Driver running on the client machine.

### Cluster Mode

In Cluster Mode, both the **Driver** and **Executors** run inside the cluster. The Cluster Manager launches the Driver on one of the worker nodes, making the application more reliable and suitable for production environments.

---

## Difference between Client Mode and Cluster Mode

| Client Mode | Cluster Mode |
|--------------|--------------|
| Driver runs on the client machine | Driver runs inside the cluster |
| Suitable for development and testing | Suitable for production environments |
| Requires a stable client connection | Continues running even if the client disconnects |
| Easier to debug | Better fault tolerance and scalability |

## Insight

Client Mode is mainly used during development because it is easier to debug, while Cluster Mode is preferred for production since the Driver runs inside the cluster, making the application more reliable and fault tolerant.

# Q14. Write a query to filter a dataset for rows where the `region` is **'North'** OR the `priority` is **'High'**.

## Answer

The `filter()` function can combine multiple conditions using the OR (`|`) operator. Since our `orders.csv` dataset does not contain the columns `region` and `priority`, we demonstrate the correct PySpark syntax by first creating these columns.

In [17]:
# Import required functions
from pyspark.sql.functions import lit

# Add sample 'region' and 'priority' columns for demonstration
df_orders = df_orders.withColumn("region", lit("North")) \
                     .withColumn("priority", lit("High"))

# Filter rows where region is 'North'
# OR priority is 'High'

filtered_df = df_orders.filter(
    (df_orders.region == "North") |
    (df_orders.priority == "High")
)

# Display the filtered records
filtered_df.show(5)

+---------+-------+--------------------+------------+------------+------+--------+
| order_id|user_id|          order_date|order_status|total_amount|region|priority|
+---------+-------+--------------------+------------+------------+------+--------+
|O00000001|U009310|2025-09-09 14:52:...|  processing|      689.66| North|    High|
|O00000002|U003247|2025-04-15 01:18:...|   completed|     1666.85| North|    High|
|O00000003|U007252|2025-04-27 15:37:...|  processing|      665.06| North|    High|
|O00000004|U008986|2025-10-04 20:35:...|   cancelled|       689.5| North|    High|
|O00000005|U008537|2024-11-13 08:15:...|   cancelled|       860.5| North|    High|
+---------+-------+--------------------+------------+------------+------+--------+
only showing top 5 rows


## Insight

The OR (`|`) operator allows Spark to retrieve rows that satisfy at least one of the specified conditions. Combining conditions using logical operators is a common practice in data filtering and analysis.

# Q15. When exploring a dataset, why is it safer to use `.show(5)` instead of `.collect()` on a multi-terabyte dataset?

## Answer

The `show(5)` function displays only the first five rows of a DataFrame without transferring the entire dataset to the Driver. It is useful for quickly inspecting data while using minimal memory.

The `collect()` function retrieves **all** rows from the distributed cluster and stores them in the Driver's memory. On very large datasets, this can consume excessive memory, slow down the application, or even cause an Out of Memory (OOM) error.

Therefore, `show(5)` is the safer and recommended option when exploring large datasets in Spark.

## Insight

Using `show(5)` is a Spark best practice because it displays a small sample of the data without loading the entire dataset into memory. This improves performance and prevents memory-related issues when working with large datasets.

# Conclusion

In this assignment, Apache Spark was used to perform various data processing tasks on structured datasets. The exercises covered DataFrame operations such as loading data, filtering records, selecting columns, creating new columns, reading and writing Parquet files, and exporting data in CSV format.

The assignment also explored important Spark concepts, including Transformations, Actions, Predicate Pushdown, Client Mode, and Cluster Mode. These concepts demonstrate how Spark efficiently processes large-scale data using distributed computing and lazy evaluation.

Overall, this assignment provided practical experience with PySpark and improved understanding of big data processing techniques used in real-world data engineering and analytics applications.